# Thinking SFT — Qwen3-4B-Instruct'i Düşünen Modele Dönüştürme

Bu notebook **smoke test ile kanıtlanmış** çalışır (`smoke/04_thinking_smoke.py`).

## Hedef
- **Başlangıç:** `unsloth/Qwen3-4B-Instruct-2507` (chat-tuned, düşünmüyor)
- **Veri:** `unsloth/OpenMathReasoning-mini` cot split (DeepSeek-R1 üretimi `<think>` trace'leri)
- **Sonuç:** Model `<think>...</think>` blokları üretmeyi öğrenir

## Niye Instruct (Thinking değil)?
**Demonstratif olarak en net:** `Qwen3-4B-Thinking-2507` zaten düşünüyor — SFT etkisi az görünür. Instruct'tan başlayınca **before/after farkı çıplak gözle** görünür.

## Donanım
| Kaynak | Smoke (3 step) | Production (60-200 step) |
|---|---|---|
| VRAM | 4.94 GB | ~5-6 GB |
| Süre | 12 sec | 30-90 dk |
| Loss | 0.74 → 0.60 | 0.5+ aralıklarına oturur |

**Test edildi:** RTX 4070 Ti SUPER 16GB

## İçindekiler

1. **Setup** — Imports + GPU
2. **Model — Qwen3-4B-Instruct (4-bit + LoRA r=32)**
3. **Pre-train Inference** — model henüz düşünmüyor (kanıt)
4. **Dataset — OpenMathReasoning-mini cot split**
5. **Format — `<think>` blokları görünmeli**
6. **Trainer + Masking**
7. **Training**
8. **Post-train Inference — `<think>` üretmeli**
9. **Save**
10. **Yaygın Tuzaklar**

## 1. Setup

In [ ]:
import unsloth                                              # MUTLAKA EN BAŞTA
from unsloth import FastLanguageModel
from unsloth.chat_templates import (
    get_chat_template,
    train_on_responses_only,
)
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer

print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

## 2. Model — Qwen3-4B-Instruct

**Kritik:** `qwen3-instruct` template kullanıyoruz, `qwen3-thinking` DEĞİL. Çünkü thinking template'i otomatik `<think>\n` ekler — biz sıfırdan öğretiyoruz.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-4B-Instruct-2507',
    max_seq_length = 2048,
    load_in_4bit = True,                          # QLoRA, 4 GB peak
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, lora_alpha = 32, lora_dropout = 0,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)

# qwen3-instruct (NOT qwen3-thinking — biz sifirdan ogretiyoruz)
tokenizer = get_chat_template(tokenizer, chat_template = 'qwen3-instruct')

## 3. Pre-train Inference — Model Henüz Düşünmüyor

Kanıt için: training'den **önce** model bir matematik sorusuna düz cevap veriyor, `<think>` blokları üretmiyor.

In [ ]:
test_msgs = [{'role': 'user', 'content': 'Hesapla 137 * 49 ve detayli cozumu goster.'}]
text = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)

print('=== PRE-TRAIN ===')
inputs = tokenizer(text, return_tensors='pt').to('cuda')
_ = model.generate(
    **inputs, max_new_tokens=200,
    temperature=0.7, top_p=0.8, top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

## 4. Dataset — OpenMathReasoning-mini

Unsloth'un kendi GRPO notebook'unda kullandığı dataset. **DeepSeek-R1 üretimi** `<think>` trace'leri içeriyor.

### Dataset Detay
- 19252 cot örnek (math problems + R1 solutions)
- Kolonlar: `expected_answer`, `problem`, `generated_solution`, vb.
- `generated_solution` alanı **`<think>...</think> + final answer`** içeriyor

In [ ]:
dataset = load_dataset('unsloth/OpenMathReasoning-mini', split='cot')
print(f'Total rows: {len(dataset)}')
print(f'Columns: {dataset.column_names}\n')

# Sample 0
print('--- Sample[0] ---')
print(f"Problem: {dataset[0]['problem'][:200]}")
print(f"\nGenerated solution (first 500 chars):")
print(dataset[0]['generated_solution'][:500])

## 5. Format — `<think>` Blokları Korunmalı

Dataset'i OpenAI messages format'ına çevir. Asistan içeriği `<think>...</think>` ile başlıyor — chat template bunu doğal olarak render eder.

In [ ]:
def to_messages(example):
    return {'conversations': [
        {'role': 'user',      'content': example['problem']},
        {'role': 'assistant', 'content': example['generated_solution']},
    ]}

dataset = dataset.map(to_messages, remove_columns=dataset.column_names)

def fmt(examples):
    return {'text': [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples['conversations']
    ]}
dataset = dataset.map(fmt, batched=True)

print('--- Formatted text[0] (first 800 chars) ---')
print(dataset[0]['text'][:800])

## 6. Trainer + Masking

**Production için:** `max_steps = 60` (T4 demo) veya `num_train_epochs = 1` (full).

**Smoke için:** `max_steps = 3` yeter (loss düşüyor, pipeline çalışıyor).

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,                          # demo; production: num_train_epochs=1
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = 'adamw_8bit',
        weight_decay = 0.001,
        lr_scheduler_type = 'linear',
        seed = 3407,
        report_to = 'none',
    ),
)

# Qwen3 markerlari
trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|im_start|>user\n',
    response_part    = '<|im_start|>assistant\n',
)

In [ ]:
# Masking dogrulama — unmasked label <think> ile baslamali
sample = trainer.train_dataset[0]
print('=== UNMASKED LABELS (sadece response gorunmeli, <think> ile baslamali) ===')
decoded = tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in sample['labels']
]).replace(tokenizer.pad_token, ' ')
print(decoded[:800])

## 7. Training

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_mem   = round(gpu_stats.total_memory / 1024**3, 3)
print(f'GPU = {gpu_stats.name} | start mem = {start_mem} GB / {max_mem} GB')

trainer_stats = trainer.train()

used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n{trainer_stats.metrics['train_runtime']:.2f} sec")
print(f"Train loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f'Peak VRAM: {used} GB ({used/max_mem*100:.1f}%)')

## 8. Post-train Inference — Model Düşünüyor mu?

60+ step eğitim sonrası model `<think>...</think>` blokları üretmeyi öğrenmeli.

**Not:** 3-5 step yetersiz — thinking emergence için min 60 step önerilir. Resmi GRPO notebook'u bile 100 step için "sabırlı olun" diyor.

In [ ]:
print('=== POST-TRAIN INFERENCE ===')
inputs = tokenizer(text, return_tensors='pt').to('cuda')
_ = model.generate(
    **inputs, max_new_tokens=400,
    temperature=0.7, top_p=0.8, top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

## 9. Save

In [ ]:
# A. LoRA adapter (en kucuk)
model.save_pretrained('qwen3_thinking_lora')
tokenizer.save_pretrained('qwen3_thinking_lora')
print('LoRA saved: qwen3_thinking_lora/')

# B. Merged 16-bit (vLLM/HF)
# model.save_pretrained_merged(
#     'qwen3_thinking_merged',
#     tokenizer,
#     save_method='merged_16bit',
# )

# C. GGUF (Ollama/llama.cpp)
# model.save_pretrained_gguf(
#     'qwen3_thinking_gguf',
#     tokenizer,
#     quantization_method='q4_k_m',
# )

## 10. Yaygın Tuzaklar

| # | Hata | Sonuç | Çözüm |
|---|---|---|---|
| 1 | `qwen3-thinking` template kullanmak | Auto `<think>\n` ekler — sen öğretmiyorsun, model zaten yapıyormuş gibi davranıyor | `qwen3-instruct` kullan |
| 2 | 3-10 step çalıştırıp "çalışmadı" demek | Thinking emergence için 60+ step lazım | En az 60-100 step yap |
| 3 | `Qwen3-4B-Thinking-2507`'tan başlamak | Demo etkisi az — model zaten düşünüyor | `Qwen3-4B-Instruct-2507`'tan başla |
| 4 | Dataset'i sadece `problem` ile vermek | `<think>` blok yok, model normal SFT yapıyor | `generated_solution` (= think + answer) ver |
| 5 | `formatting_prompts_func`'ta `add_generation_prompt=True` | Çift assistant header | Train'de **False**, inference'ta True |
| 6 | Türkçe veride thinking beklemek | Model R1 traces'i ezbere | Dataset Türkçeleştir veya GPT-4'le translate et |

## Özet

1. **Pipeline aynı normal SFT** (`01_sft_modern.ipynb` ile birebir benzer)
2. Tek fark: **dataset'te `<think>` trace'leri var**, bu yüzden model thinking'i öğreniyor
3. **`qwen3-instruct` template kullan** — `qwen3-thinking` otomatik `<think>` ekler, kendin öğretmek istiyorsan instruct seç
4. **Min 60 step** thinking emergence için — 3 step pipeline kontrolü, gerçek demo için yetmez
5. **VRAM 4-5 GB** — 16GB'da çok rahat
6. **GRPO bir adım daha ileri** — bu SFT'den sonra reasoning kalitesi artırılabilir

**Test edildi:** `smoke/04_thinking_smoke.py` — pipeline tüm cell'leri ile RTX 4070 Ti SUPER 16GB'da geçti.